In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [2]:
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

emulator_1_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_1/predictions_4d.zarr'
emulator_1_patch_full = xr.open_dataset(emulator_1_path, consolidated=True) 

emulator_2_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_2/predictions_4d.zarr'
emulator_2_patch_full = xr.open_dataset(emulator_2_path, consolidated=True) 

emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_3/predictions_4d.zarr'
emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_4/predictions_4d.zarr'
emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

In [3]:
emulator_times = emulator_1_patch_full.time.values
matching_times = pd.DatetimeIndex([
    pd.Timestamp(t.year, t.month, t.day, t.hour, t.minute, t.second)
    for t in emulator_times
])

llc_patch = llc_patch_full.sel(time=matching_times)

In [4]:

# emulator_1_patch = emulator_1_patch_full.isel(time=slice(0, extent))
# emulator_2_patch = emulator_2_patch_full.isel(time=slice(0, extent))
# emulator_3_patch = emulator_3_patch_full.isel(time=slice(0, extent))
# emulator_4_patch = emulator_4_patch_full.isel(time=slice(0, extent))

emulator_1_patch = emulator_1_patch_full
emulator_2_patch = emulator_2_patch_full
emulator_3_patch = emulator_3_patch_full
emulator_4_patch = emulator_4_patch_full

In [5]:
# ============== SET VARIABLES HERE ==============
vars = ['Theta', 'Salt', 'U', 'V']  # Add more variables as needed
# ================================================

def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        # cftime objects
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        # numpy datetime64
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

for var in vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_times = len(emulator_1_patch.time)
    time_step = 1
    time_indices = list(range(0, n_times, time_step))
    nrows = len(time_indices)
    ncols = 5
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        vmin = np.min([llc_vis.values.min(), emulator_1_vis.values.min(), 
                       emulator_2_vis.values.min(), emulator_3_vis.values.min(),
                       emulator_4_vis.values.min()])
        vmax = np.max([llc_vis.values.max(), emulator_1_vis.values.max(), 
                       emulator_2_vis.values.max(), emulator_3_vis.values.max(),
                       emulator_4_vis.values.max()])
        
        ax1, ax2, ax3, ax4, ax5 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3], axes[row, 4]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), llc_vis, 
                           cmap="Spectral_r", vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC {var} {time_str}', fontsize=8)
        plt.colorbar(cf1, ax=ax1)
        
        cf2 = ax2.contourf(emulator_1_vis.coords.get('i', np.arange(emulator_1_vis.shape[-1])), 
                           emulator_1_vis.coords.get('j', np.arange(emulator_1_vis.shape[-2])), emulator_1_vis, 
                           cmap="Spectral_r", vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'Emulator 1 {var} {time_str}', fontsize=8)
        plt.colorbar(cf2, ax=ax2)
        
        cf3 = ax3.contourf(emulator_2_vis.coords.get('i', np.arange(emulator_2_vis.shape[-1])), 
                           emulator_2_vis.coords.get('j', np.arange(emulator_2_vis.shape[-2])), emulator_2_vis, 
                           cmap="Spectral_r", vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'Emulator 2 {var} {time_str}', fontsize=8)
        plt.colorbar(cf3, ax=ax3)
        
        cf4 = ax4.contourf(emulator_3_vis.coords.get('i', np.arange(emulator_3_vis.shape[-1])), 
                           emulator_3_vis.coords.get('j', np.arange(emulator_3_vis.shape[-2])), emulator_3_vis, 
                           cmap="Spectral_r", vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'Emulator 3 {var} {time_str}', fontsize=8)
        plt.colorbar(cf4, ax=ax4)
        
        cf5 = ax5.contourf(emulator_4_vis.coords.get('i', np.arange(emulator_4_vis.shape[-1])), 
                           emulator_4_vis.coords.get('j', np.arange(emulator_4_vis.shape[-2])), emulator_4_vis, 
                           cmap="Spectral_r", vmin=vmin, vmax=vmax, levels=30)
        ax5.set_title(f'Emulator 4 {var} {time_str}', fontsize=8)
        plt.colorbar(cf5, ax=ax5)
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols-1, figsize=(16, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        diff_1 = llc_vis.values - emulator_1_vis.values
        diff_2 = llc_vis.values - emulator_2_vis.values
        diff_3 = llc_vis.values - emulator_3_vis.values
        diff_4 = llc_vis.values - emulator_4_vis.values
        
        abs_max = np.max([np.abs(diff_1).max(), np.abs(diff_2).max(), 
                          np.abs(diff_3).max(), np.abs(diff_4).max()])
        vmin, vmax = -abs_max, abs_max
        
        ax1, ax2, ax3, ax4 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_1, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC - Em1 {var} {time_str}', fontsize=8)
        
        cf2 = ax2.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_2, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'LLC - Em2 {var} {time_str}', fontsize=8)
        
        cf3 = ax3.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_3, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'LLC - Em3 {var} {time_str}', fontsize=8)
        
        cf4 = ax4.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_4, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'LLC - Em4 {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf4, ax=[ax1, ax2, ax3, ax4], orientation='vertical', 
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"Saved plots for {var}")

Generating plots for Theta...
Saved plots for Theta
Generating plots for Salt...
Saved plots for Salt
Generating plots for U...
Saved plots for U
Generating plots for V...
Saved plots for V
